# DCASE2026 Task1 Colab: train with audio+text, test with audio only

This notebook trains the baseline `BaseClassifier` with `mode="both"`, then evaluates the trained checkpoint with audio-only inference by switching the loaded model to `mode="audio"` at test time.

Important: this is an intentional modality-ablation test. The model is trained with audio+text fusion, but the test forward pass skips the text branch.

In [ ]:
# 1. Mount Drive and imports
from google.colab import drive
drive.mount('/content/drive')

import json
import os
import random
import shutil
import time
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())

In [ ]:
# 2. Paths and baseline-like settings
METADATA_CSV = '/content/drive/MyDrive/DCASE2026/data/metadata/BSD10k_metadata.csv'
DRIVE_AUDIO_EMB_DIR = '/content/drive/MyDrive/DCASE2026/data/features/clap_audio_embeddings'
DRIVE_TEXT_EMB_DIR  = '/content/drive/MyDrive/DCASE2026/data/features/clap_text_embeddings'

LOCAL_BASE_DIR = '/content/bsd10k_features_input_audio_test'
AUDIO_EMB_DIR = f'{LOCAL_BASE_DIR}/clap_audio_embeddings'
TEXT_EMB_DIR = f'{LOCAL_BASE_DIR}/clap_text_embeddings'

OUTPUT_DIR = Path('/content/drive/MyDrive/DCASE2026/outputs/input_audio_dcase2026_task1_colab')
DATA_DIR = OUTPUT_DIR / 'data'
MODEL_OUTPUT_DIR = OUTPUT_DIR / 'model_output'
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 1821
N_FOLDS = 5
BATCH_SIZE = 64
NUM_EPOCHS = 100
LEARNING_RATE = 0.001
PATIENCE = 5
EARLY_STOPPING_FACTOR = 3
TRAIN_MODE = 'both'
TEST_MODE = 'audio'

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print('output:', OUTPUT_DIR)

In [ ]:
# 3. Copy Drive embeddings to Colab local disk for faster/stable np.load
def copy_one_feature_dir(src, dst, retries=3):
    src = Path(src)
    dst = Path(dst)
    dst.mkdir(parents=True, exist_ok=True)

    for attempt in range(1, retries + 1):
        try:
            if not src.exists():
                raise FileNotFoundError(f'Drive source folder is not visible: {src}')
            files = sorted(src.glob('*.npy'))
            if len(files) == 0:
                raise RuntimeError(f'No .npy files found in {src}. Remount Google Drive and retry.')
            copied = 0
            skipped = 0
            print(f'Copying {len(files):,} files: {src} -> {dst} (attempt {attempt}/{retries})')
            for file_path in files:
                out_path = dst / file_path.name
                if out_path.exists() and out_path.stat().st_size == file_path.stat().st_size:
                    skipped += 1
                    continue
                shutil.copy2(file_path, out_path)
                copied += 1
                if copied % 1000 == 0:
                    print(f'  copied {copied:,}, skipped {skipped:,}')
            print(f'Done: copied {copied:,}, skipped {skipped:,}, local total {len(list(dst.glob("*.npy"))):,}')
            return
        except Exception as exc:
            print(f'copy failed on attempt {attempt}: {repr(exc)}')
            if attempt < retries:
                time.sleep(10)
            else:
                raise

copy_one_feature_dir(DRIVE_AUDIO_EMB_DIR, AUDIO_EMB_DIR)
copy_one_feature_dir(DRIVE_TEXT_EMB_DIR, TEXT_EMB_DIR)

In [ ]:
# 4. Build processed dataset like dcase2026_task1_baseline/build_dataset.py
def build_processed_dataset():
    df = pd.read_csv(METADATA_CSV)
    df['sound_id'] = df['sound_id'].astype(str).str.strip()

    print('Original rows:', len(df))
    print('Original classes:', df['class'].nunique())

    # Match baseline filtering: discard top-level and '-other' style category ids ending 00/99 with length 3.
    s = df['class_idx'].astype(str)
    df = df[~((s.str.len() == 3) & (s.str.endswith('99') | s.str.endswith('00')))].copy()
    print('After class_idx filtering:', len(df))

    df['original_class_idx'] = df['class_idx']
    original_indices = sorted(df['original_class_idx'].unique())
    index_mapping = {orig: new for new, orig in enumerate(original_indices)}
    df['class_idx'] = df['original_class_idx'].map(index_mapping)
    df['class_top'] = df['class'].apply(lambda x: x.split('-')[0] if isinstance(x, str) else None)

    df_sorted = df.sort_values('original_class_idx')
    top_classes = df_sorted['class_top'].drop_duplicates()
    top_class_dict = {cls: i for i, cls in enumerate(top_classes)}
    df['top_class_idx'] = df['class_top'].map(top_class_dict)
    class_dict = dict(zip(df['class'], df['class_idx']))

    records = []
    for _, row in df.iterrows():
        sid = str(row['sound_id']).strip()
        audio_path = Path(AUDIO_EMB_DIR) / f'{sid}.npy'
        text_path = Path(TEXT_EMB_DIR) / f'{sid}.npy'
        if not audio_path.is_file() or not text_path.is_file():
            continue
        records.append({
            'index': sid,
            'audio_emb_filepath': str(audio_path),
            'text_emb_filepath': str(text_path),
            'top_class': row['class_top'],
            'top_class_idx': int(row['top_class_idx']),
            'class': row['class'],
            'class_idx': int(row['class_idx']),
        })

    processed = pd.DataFrame(records)
    processed.to_csv(DATA_DIR / 'processed_dataset.csv', index=False)
    with open(DATA_DIR / 'class_dict.json', 'w') as f:
        json.dump(class_dict, f, indent=2)
    with open(DATA_DIR / 'top_class_dict.json', 'w') as f:
        json.dump(top_class_dict, f, indent=2)
    return processed, class_dict, top_class_dict

full_df, class_dict, top_class_dict = build_processed_dataset()
id_to_class = {int(v): k for k, v in class_dict.items()}
class_names = [id_to_class[i] for i in range(len(id_to_class))]
print('Processed rows:', len(full_df))
print('Classes:', full_df['class_idx'].nunique())
display(full_df.head())

In [ ]:
# 5. Dataset, model, loss
class HATRDataset(Dataset):
    def __init__(self, dataframe, aug=True, mask_pct=0.7):
        self.dataframe = dataframe.reset_index(drop=True)
        self.aug = aug
        self.mask_pct = mask_pct

    def __len__(self):
        return len(self.dataframe)

    def _rand_mask(self, emb):
        max_mask = max(1, int(emb.shape[0] * self.mask_pct))
        num_to_mask = random.randint(1, max_mask)
        mask_indices = torch.randperm(emb.shape[0])[:num_to_mask]
        mask = torch.ones_like(emb)
        mask[mask_indices] = 0.0
        return emb * mask

    def __getitem__(self, idx):
        sample = self.dataframe.iloc[idx]
        audio_emb = torch.tensor(np.load(sample['audio_emb_filepath']).reshape(-1), dtype=torch.float32)
        text_emb = torch.tensor(np.load(sample['text_emb_filepath']).reshape(-1), dtype=torch.float32)
        if self.aug:
            audio_emb = self._rand_mask(audio_emb + torch.randn_like(audio_emb) * 0.0001)
            text_emb = self._rand_mask(text_emb + torch.randn_like(text_emb) * 0.0001)
        return {
            'sound_id': sample['index'],
            'audio_embedding': audio_emb,
            'text_embedding': text_emb,
            'class_idx': int(sample['class_idx']),
            'top_class_idx': int(sample['top_class_idx']),
        }

class ResidualBlock(nn.Module):
    def __init__(self, input_size, hidden_size, dropout=0.2, use_batch_norm=True):
        super().__init__()
        self.use_batch_norm = use_batch_norm
        self.linear1 = nn.Linear(input_size, hidden_size)
        self.linear2 = nn.Linear(hidden_size, input_size)
        self.activation = nn.LeakyReLU()
        self.dropout = nn.Dropout(dropout)
        if use_batch_norm:
            self.norm1 = nn.BatchNorm1d(hidden_size)
            self.norm2 = nn.BatchNorm1d(input_size)

    def forward(self, x):
        residual = x
        out = self.linear1(x)
        if self.use_batch_norm:
            out = self.norm1(out)
        out = self.activation(out)
        out = self.dropout(out)
        out = self.linear2(out)
        if self.use_batch_norm:
            out = self.norm2(out)
        return self.activation(out + residual)

class EmbeddingEncoder(nn.Module):
    def __init__(self, input_size, output_size, dropout=0.2, use_batch_norm=True, num_residual_blocks=3):
        super().__init__()
        hidden_size = max(input_size, output_size * 2)
        self.input_projection = nn.Sequential(nn.Linear(input_size, hidden_size), nn.LeakyReLU(), nn.Dropout(dropout))
        self.residual_blocks = nn.ModuleList([ResidualBlock(hidden_size, hidden_size * 2, dropout, use_batch_norm) for _ in range(num_residual_blocks)])
        self.output_projection = nn.Sequential(nn.Linear(hidden_size, hidden_size // 2), nn.LeakyReLU(), nn.Dropout(dropout), nn.Linear(hidden_size // 2, output_size))
        self.use_batch_norm = use_batch_norm
        if use_batch_norm:
            self.input_norm = nn.BatchNorm1d(input_size)
            self.output_norm = nn.BatchNorm1d(output_size)

    def forward(self, x):
        if self.use_batch_norm:
            x = self.input_norm(x)
        x = self.input_projection(x)
        for block in self.residual_blocks:
            x = block(x)
        x = self.output_projection(x)
        if self.use_batch_norm:
            x = self.output_norm(x)
        return x

class AttentionFusion(nn.Module):
    def __init__(self, feature_size, dropout=0.2):
        super().__init__()
        self.attention = nn.Sequential(nn.Linear(feature_size * 2, feature_size), nn.Tanh(), nn.Linear(feature_size, 2), nn.Softmax(dim=-1))
        self.dropout = nn.Dropout(dropout)

    def forward(self, audio_features, text_features):
        weights = self.attention(torch.cat([audio_features, text_features], dim=-1))
        fused = audio_features * weights[:, 0:1] + text_features * weights[:, 1:2]
        return self.dropout(fused), weights

class BaseClassifier(nn.Module):
    def __init__(self, hidden_size=128, num_classes=23, emb_size_audio=512, emb_size_text=512, dropout=0.1, use_batch_norm=True, mode='both', num_residual_blocks=3, use_attention_fusion=True):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_classes = num_classes
        self.emb_size_audio = emb_size_audio
        self.emb_size_text = emb_size_text
        self.dropout = dropout
        self.use_batch_norm = use_batch_norm
        self.mode = mode
        self.use_attention_fusion = use_attention_fusion and mode == 'both'
        self.audio_emb_extractor = EmbeddingEncoder(emb_size_audio, hidden_size, dropout, use_batch_norm, num_residual_blocks) if mode in ['audio', 'both'] else None
        self.text_emb_extractor = EmbeddingEncoder(emb_size_text, hidden_size, dropout, use_batch_norm, num_residual_blocks) if mode in ['text', 'both'] else None
        combined_size = hidden_size if mode != 'both' or self.use_attention_fusion else hidden_size * 2
        self.fusion = AttentionFusion(hidden_size, dropout) if self.use_attention_fusion else None
        self.latent_projector = nn.Sequential(nn.Linear(combined_size, hidden_size * 2), nn.LeakyReLU(), nn.Dropout(dropout), nn.Linear(hidden_size * 2, hidden_size), nn.LeakyReLU(), nn.Dropout(dropout), nn.Linear(hidden_size, hidden_size // 2), nn.LeakyReLU(), nn.Dropout(dropout / 2))
        self.residual_classifier = nn.ModuleList([ResidualBlock(hidden_size // 2, hidden_size, dropout / 2, use_batch_norm) for _ in range(2)])
        self.class_predictor = nn.Sequential(nn.Linear(hidden_size // 2, hidden_size // 4), nn.LeakyReLU(), nn.Dropout(dropout / 4), nn.Linear(hidden_size // 4, num_classes))

    def forward(self, audio_emb=None, text_emb=None):
        features = []
        if self.mode in ['audio', 'both']:
            features.append(self.audio_emb_extractor(audio_emb))
        if self.mode in ['text', 'both']:
            features.append(self.text_emb_extractor(text_emb))
        if len(features) > 1:
            if self.use_attention_fusion:
                combined, attn = self.fusion(features[0], features[1])
            else:
                combined, attn = torch.cat(features, dim=-1), None
        else:
            combined, attn = features[0], None
        z = self.latent_projector(combined)
        for block in self.residual_classifier:
            z = block(z)
        return z, self.class_predictor(z), attn

criterion = nn.CrossEntropyLoss(label_smoothing=0.01)

def init_weights(module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

In [ ]:
# 6. Metrics and helpers
def build_class_to_topclass_mapping(class_dict, top_class_dict):
    mapping = {}
    for class_name, class_id in class_dict.items():
        top_name = class_name.split('-')[0]
        mapping[int(class_id)] = int(top_class_dict[top_name])
    return mapping

class_to_top_class = build_class_to_topclass_mapping(class_dict, top_class_dict)

def extend_subcat(label_name):
    parts = str(label_name).split('-')
    return [parts[0], '-'.join(parts[1:]) if len(parts) > 1 else '']

def get_top_level(label_name):
    return str(label_name).split('-')[0]

def hierarchical_accuracy_for_class(subcat, pairs, lambda_param=0.5):
    scores = []
    for pred, gt in pairs:
        if subcat == gt:
            pred_top, pred_sub = extend_subcat(pred)
            gt_top, gt_sub = extend_subcat(gt)
            scores.append(1.0 if pred == gt else lambda_param if pred_top == gt_top else 0.0)
    return np.mean(scores) if scores else np.nan

def hierarchical_prf_weighted_for_class(subcat, pairs, lambda_param=0.75):
    hPP, hRR = [], []
    for pred, gt in pairs:
        pi, ti = extend_subcat(pred), extend_subcat(gt)
        overlap = len(set(pi).intersection(set(ti)))
        w = 1.0 if pred == gt else lambda_param if get_top_level(pred) == get_top_level(gt) else 0.0
        if subcat == pred:
            hPP.append((w * overlap) / len(pi))
        if subcat == gt:
            hRR.append((w * overlap) / len(ti))
    p = np.mean(hPP) if hPP else np.nan
    r = np.mean(hRR) if hRR else np.nan
    if np.isnan(p) or np.isnan(r):
        f = np.nan
    elif p == 0 and r == 0:
        f = 0.0
    else:
        f = 2 * p * r / (p + r)
    return p, r, f

def compute_metrics(y_true, y_pred):
    y_true = [int(x) for x in y_true]
    y_pred = [int(x) for x in y_pred]
    gt_labels = [id_to_class[x] for x in y_true]
    pred_labels = [id_to_class[x] for x in y_pred]
    pairs = list(zip(pred_labels, gt_labels))
    classes = sorted(set(gt_labels))
    class_accs, class_top_accs = [], []
    for c in sorted(set(y_true)):
        idx = [i for i, gt in enumerate(y_true) if gt == c]
        class_accs.append(np.mean([y_pred[i] == y_true[i] for i in idx]))
        class_top_accs.append(np.mean([class_to_top_class.get(y_true[i]) == class_to_top_class.get(y_pred[i]) for i in idx]))
    h_accs, hPs, hRs, hFs = [], [], [], []
    for c in classes:
        h_acc = hierarchical_accuracy_for_class(c, pairs, lambda_param=0.5)
        if not np.isnan(h_acc): h_accs.append(h_acc)
        hP, hR, hF = hierarchical_prf_weighted_for_class(c, pairs, lambda_param=0.75)
        if not np.isnan(hP): hPs.append(hP)
        if not np.isnan(hR): hRs.append(hR)
        if not np.isnan(hF): hFs.append(hF)
    return {
        'accuracy': 100 * np.mean(np.array(y_true) == np.array(y_pred)),
        'top_accuracy': 100 * np.mean([class_to_top_class.get(gt) == class_to_top_class.get(pred) for gt, pred in zip(y_true, y_pred)]),
        'macro_accuracy': 100 * np.mean(class_accs),
        'macro_top_accuracy': 100 * np.mean(class_top_accs),
        'hierarchical_accuracy': 100 * np.mean(h_accs),
        'hierarchical_precision': 100 * np.mean(hPs),
        'hierarchical_recall': 100 * np.mean(hRs),
        'hierarchical_f1': 100 * np.mean(hFs),
    }

def make_loader(df, aug=False, shuffle=False, drop_last=False):
    ds = HATRDataset(df, aug=aug, mask_pct=0.7)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, drop_last=drop_last, num_workers=0, pin_memory=torch.cuda.is_available())

In [ ]:
# 7. Train both, evaluate audio-only
def make_model(mode=TRAIN_MODE):
    model = BaseClassifier(
        hidden_size=128,
        num_classes=len(class_dict),
        emb_size_audio=512 if mode in ['audio', 'both'] else 0,
        emb_size_text=512 if mode in ['text', 'both'] else 0,
        dropout=0.1,
        use_batch_norm=True,
        mode=mode,
    ).to(device)
    model.apply(init_weights)
    return model

def train_epoch(model, loader, optimizer):
    model.train()
    total_loss, total, correct = 0.0, 0, 0
    for batch in loader:
        labels = batch['class_idx'].long().to(device)
        audio = batch['audio_embedding'].to(device)
        text = batch['text_embedding'].to(device)
        optimizer.zero_grad()
        _, logits, _ = model(audio, text)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += float(loss.detach().cpu()) * labels.size(0)
        total += labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
    return total_loss / total, correct / total

@torch.no_grad()
def predict(model, loader, eval_mode='both'):
    old_mode = model.mode
    model.mode = eval_mode
    model.eval()
    y_true, y_pred = [], []
    total_loss, total = 0.0, 0
    for batch in loader:
        labels = batch['class_idx'].long().to(device)
        audio = batch['audio_embedding'].to(device)
        text = batch['text_embedding'].to(device) if eval_mode in ['text', 'both'] else None
        _, logits, _ = model(audio, text)
        loss = criterion(logits, labels)
        total_loss += float(loss.detach().cpu()) * labels.size(0)
        total += labels.size(0)
        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(logits.argmax(1).cpu().numpy().tolist())
    model.mode = old_mode
    return total_loss / total, np.array(y_true), np.array(y_pred)

def train_one_fold(fold, train_df, val_df, test_df):
    fold_dir = MODEL_OUTPUT_DIR / f'fold_{fold}'
    fold_dir.mkdir(parents=True, exist_ok=True)
    train_loader = make_loader(train_df, aug=True, shuffle=True, drop_last=True)
    val_loader = make_loader(val_df, aug=False)
    test_loader = make_loader(test_df, aug=False)
    model = make_model(TRAIN_MODE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
    best_val_acc = -1.0
    stale = 0
    history = []
    best_path = fold_dir / 'best_both_model.pth'

    for epoch in tqdm(range(1, NUM_EPOCHS + 1), desc=f'Fold {fold}'):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer)
        val_loss, yv, pv = predict(model, val_loader, eval_mode=TRAIN_MODE)
        val_acc = np.mean(yv == pv)
        scheduler.step()
        history.append({'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc, 'val_loss': val_loss, 'val_acc': val_acc})
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            stale = 0
            torch.save({'model_state': model.state_dict(), 'config': {'mode': TRAIN_MODE, 'hidden_size': 128, 'num_classes': len(class_dict), 'emb_size_audio': 512, 'emb_size_text': 512, 'dropout': 0.1, 'use_batch_norm': True}}, best_path)
        else:
            stale += 1
        if epoch == 1 or epoch % 10 == 0:
            print(f'[{epoch:03d}/{NUM_EPOCHS}] train_acc={train_acc:.4f} val_acc={val_acc:.4f}')
        if stale >= PATIENCE * EARLY_STOPPING_FACTOR:
            print(f'Early stopping at epoch {epoch}')
            break

    pd.DataFrame(history).to_csv(fold_dir / 'history.csv', index=False)
    checkpoint = torch.load(best_path, map_location=device)
    model.load_state_dict(checkpoint['model_state'])

    test_loss_audio, y_true_audio, y_pred_audio = predict(model, test_loader, eval_mode=TEST_MODE)
    audio_metrics = compute_metrics(y_true_audio, y_pred_audio)
    cm_audio = confusion_matrix(y_true_audio, y_pred_audio, labels=list(range(len(class_dict))))
    np.save(fold_dir / 'confusion_matrix_audio_test.npy', cm_audio)
    pd.DataFrame({'y_true': y_true_audio, 'y_pred': y_pred_audio}).to_csv(fold_dir / 'predictions_audio_test.csv', index=False)
    pd.DataFrame([audio_metrics]).to_csv(fold_dir / 'metrics_audio_test.csv', index=False)

    print(f'Fold {fold} audio-test accuracy={audio_metrics["accuracy"]:.2f}% hierarchical_f1={audio_metrics["hierarchical_f1"]:.2f}%')
    return {'fold': fold, 'best_val_accuracy_both': 100 * best_val_acc, 'cm_audio': cm_audio, **audio_metrics}


In [ ]:
# 8. Run 5-fold experiment
labels = full_df['class_idx'].astype(int).to_numpy()
class_counts = pd.Series(labels).value_counts()
if int(class_counts.min()) < N_FOLDS:
    raise ValueError(f'Rarest class has {int(class_counts.min())} samples, cannot run {N_FOLDS}-fold StratifiedKFold.')

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_results = []

for fold, (trainval_idx, test_idx) in enumerate(skf.split(np.zeros(len(labels)), labels), start=1):
    trainval_labels = labels[trainval_idx]
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    train_rel, val_rel = next(sss.split(np.zeros(len(trainval_labels)), trainval_labels))
    train_idx = trainval_idx[train_rel]
    val_idx = trainval_idx[val_rel]
    train_df = full_df.iloc[train_idx].reset_index(drop=True)
    val_df = full_df.iloc[val_idx].reset_index(drop=True)
    test_df = full_df.iloc[test_idx].reset_index(drop=True)
    print(f'\n==== Fold {fold}/{N_FOLDS} | train both, test audio ====', flush=True)
    print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')
    fold_results.append(train_one_fold(fold, train_df, val_df, test_df))

summary = pd.DataFrame([{k: v for k, v in r.items() if k != 'cm_audio'} for r in fold_results])
summary.to_csv(MODEL_OUTPUT_DIR / 'summary_audio_test.csv', index=False)
display(summary)

metric_cols = ['accuracy', 'top_accuracy', 'macro_accuracy', 'macro_top_accuracy', 'hierarchical_accuracy', 'hierarchical_precision', 'hierarchical_recall', 'hierarchical_f1']
mean_std = pd.DataFrame({'mean': summary[metric_cols].mean(), 'std': summary[metric_cols].std(ddof=0)})
mean_std['mean_pm_std'] = mean_std.apply(lambda r: f'{r["mean"]:.2f}% ± {r["std"]:.2f}%', axis=1)
mean_std.to_csv(MODEL_OUTPUT_DIR / 'summary_audio_test_mean_std.csv')
display(mean_std)

In [ ]:
# 9. Mean ± std audio-test confusion matrix, baseline-style
cms = np.stack([r['cm_audio'] for r in fold_results], axis=0).astype(float)
row_sums = cms.sum(axis=2, keepdims=True)
cm_norms = np.divide(cms, row_sums, out=np.zeros_like(cms), where=row_sums != 0)
cm_mean = cm_norms.mean(axis=0)
cm_std = cm_norms.std(axis=0, ddof=0)

pd.DataFrame(cm_mean, index=class_names, columns=class_names).to_csv(MODEL_OUTPUT_DIR / 'confusion_matrix_audio_test_mean_normalized_true.csv')
pd.DataFrame(cm_std, index=class_names, columns=class_names).to_csv(MODEL_OUTPUT_DIR / 'confusion_matrix_audio_test_std_normalized_true.csv')

def annotate_mean_std(ax, mean_matrix, std_matrix, fontsize=5):
    for i in range(mean_matrix.shape[0]):
        for j in range(mean_matrix.shape[1]):
            v = float(mean_matrix[i, j])
            s = float(std_matrix[i, j])
            color = 'white' if v >= 0.45 else 'black'
            stroke = 'black' if color == 'white' else 'white'
            text = ax.text(j, i, f'{v:.2f}\n±{s:.2f}', ha='center', va='center', fontsize=fontsize, color=color)
            text.set_path_effects([path_effects.withStroke(linewidth=1.15, foreground=stroke)])

n = len(class_names)
fig_size = max(10, n * 0.55)
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
im = ax.imshow(cm_mean, interpolation='nearest', cmap='Blues', vmin=0.0, vmax=1.0)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Normalized true-class ratio', rotation=270, labelpad=18)
ax.set_title('Train both, test audio only | mean ± std normalized confusion matrix')
ax.set_xlabel('Predicted class')
ax.set_ylabel('True class')
ax.set_xticks(np.arange(n))
ax.set_yticks(np.arange(n))
ax.set_xticklabels(class_names, rotation=90, fontsize=8)
ax.set_yticklabels(class_names, fontsize=8)
annotate_mean_std(ax, cm_mean, cm_std, fontsize=5 if n > 20 else 6)
fig.tight_layout()
save_path = MODEL_OUTPUT_DIR / 'confusion_matrix_audio_test_mean_pm_std_normalized_true.png'
fig.savefig(save_path, dpi=220, bbox_inches='tight')
plt.show()
print('saved:', save_path)